# 23 — Memory-Augmented RAG

> **Tier 5 | Multi-turn & Memory**

## What is Memory-Augmented RAG?

Standard RAG treats every question as independent. **Memory-Augmented RAG** maintains a
**conversation history per session** and uses it to:

1. **Resolve pronoun references** — "it", "they", "this" → expanded with entity from prior turn
2. **Enrich retrieval** — the resolved query embeds richer context, surfacing more relevant chunks
3. **Ground generation** — the LLM receives both retrieved chunks *and* recent conversation turns

### Memory components

| Component | Role |
|-----------|------|
| `ConversationMemory` | Session-keyed store of `{role, content}` turns |
| `resolve_query()` | Detects pronouns → prepends prior context to query |
| `memory_augmented_rag()` | Full pipeline: resolve → retrieve → generate → store |

## PDF Framework: pypdf (`PdfReader`)

This notebook uses **pypdf** with `PdfReader.pages[i].extract_text()` — the simplest
available API. No layout analysis or block detection: just clean text per page.

| Feature | pypdf |
|---------|-------|
| API | `PdfReader(path).pages[i].extract_text()` |
| Layout awareness | None — plain text only |
| Speed | Very fast |
| Use case | Clean text PDFs with no table/image extraction needs |

## PDF Framework Progression (Qdrant series)

| NB | Pattern | Framework | Key API |
|----|---------|-----------|----------|
| 18 | Corrective RAG | **pdfplumber** | `bbox_density` |
| 19 | Self RAG | **pymupdf** `get_text("dict")` | Font size/bold → headings |
| 20 | Iterative RAG | **pdfminer.six** | `LTTextBox.y0` zone tagging |
| 21 | Recursive RAG | **pdfminer.six** heuristic | Element-type hierarchy |
| 22 | Agentic RAG | **pymupdf** `get_text("blocks")` | Block-type filter, block_no |
| **23** | **Memory-Augmented RAG** | **pypdf** `PdfReader` | `page.extract_text()` |
| 24 | Multi-Turn Conversational RAG | TBD | |

## Flow Diagram


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pypdf PdfReader"]
        PDF(["📄 climate.pdf"])
        PDF --> PY["pypdf.PdfReader\npage.extract_text()"]
        PY --> CH["Text chunks\n(page-level split)"]
        CH --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\nmemory_rag_23")]
    end
    subgraph MEMORY ["🧠  CONVERSATION MEMORY"]
        MEM["ConversationMemory\n(session store)"]
        MEM --> RES["resolve()\nPronoun detection\n+ context injection"]
    end
    subgraph RAG ["🤖  MEMORY-AUGMENTED RAG"]
        Q(["❓ Query"])
        Q --> RES
        RES --> EMB2["Embed resolved query"]
        EMB2 --> QDRANT
        QDRANT --> CTX["Retrieved chunks"]
        CTX --> HIST["+ Conversation history"]
        HIST --> GEN["LLM Generation\nClaude Sonnet 4.6"]
        GEN --> ANS(["✅ Answer"])
        ANS --> MEM
    end
    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style MEMORY fill:#fef3c7,stroke:#f59e0b,color:#78350f
    style RAG fill:#fdf4ff,stroke:#9333ea,color:#3b0764
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "pypdf", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid, re
from typing import List, Dict
from dotenv import load_dotenv

import boto3
import pypdf
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

COLLECTION_NAME = "memory_rag_23"
EMBEDDING_DIM   = 1024
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
DEFAULT_TOP_K   = 5
MEMORY_WINDOW   = 6   # max recent turns kept in context

PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection    : {COLLECTION_NAME}")
print(f"PDF           : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")
print(f"Memory window : {MEMORY_WINDOW} turns")


## Step 3 — Vector Store

In [ ]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name = collection_name
        self._chunks: List[Dict] = []
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}'); return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}'); return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
        if exists and force_recreate:
            self._qdrant.delete_collection(self.name); exists = False
        if not exists:
            self._qdrant.create_collection(
                self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
            print(f'Created "{self.name}" (dim={dim})')
        else:
            print(f'"{self.name}" already exists')

    def upsert(self, docs: List[Dict]):
        pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                           payload={'text': d['text'], 'metadata': d.get('metadata', {})})
               for d in docs]
        self._qdrant.upsert(collection_name=self.name, points=pts)
        self._chunks.extend(docs)

    def vector_search(self, qvec: List[float], top_k: int = 5) -> List[Dict]:
        resp = self._qdrant.query_points(
            collection_name=self.name, query=qvec,
            limit=top_k, with_payload=True)
        return [{'text': p.payload.get('text',''),
                 'metadata': p.payload.get('metadata',{}),
                 'score': p.score} for p in resp.points]

    def count(self) -> int:
        return self._qdrant.get_collection(self.name).points_count or 0

print("VectorStore defined.")


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

def call_llm(system: str, user_content: str, max_tokens: int = 1024) -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system,
        "messages": [{"role": "user", "content": user_content}],
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"]

test_emb = embed_text("memory augmented rag pypdf test")
print(f"Embedding OK — dim={len(test_emb)}")
print("call_llm ready.")


## Step 5 — PDF Loading with pypdf (`PdfReader`)

`pypdf.PdfReader(path).pages[i].extract_text()` returns plain text per page.
No layout analysis — the simplest extraction API in the series.
Clean text PDFs like `climate.pdf` are ideal candidates.


In [ ]:
def recursive_split(text: str, size: int, overlap: int) -> List[str]:
    if len(text) <= size:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        end = start + size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += size - overlap
    return chunks


def load_pdf_pypdf(path: str):
    reader  = pypdf.PdfReader(path)
    n_pages = len(reader.pages)
    chunks  = []

    for page_num, page in enumerate(reader.pages):
        text = (page.extract_text() or "").strip()
        if not text:
            continue
        for sub in recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP):
            chunks.append({
                'text'      : sub,
                'page'      : page_num + 1,
                'char_count': len(sub),
            })

    stats = {
        'n_pages' : n_pages,
        'n_chunks': len(chunks),
        'avg_chars': sum(c['char_count'] for c in chunks) // max(len(chunks), 1),
    }
    return chunks, stats


t0 = time.time()
chunks, stats = load_pdf_pypdf(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"pypdf extraction  : {elapsed:.0f}ms")
print(f"Pages             : {stats['n_pages']}")
print(f"Chunks            : {stats['n_chunks']}")
print(f"Avg chunk length  : {stats['avg_chars']} chars")


## Step 6 — Connect & Index

In [ ]:
vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL, qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL, region=AWS_REGION)
print(f"Backend: {vs._backend}")

vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

print(f"Embedding {len(chunks)} chunks...")
t0   = time.time()
embs = embed_batch([c['text'] for c in chunks], label='[index]')

docs = [
    {
        'text'     : chunks[i]['text'],
        'embedding': embs[i],
        'metadata' : {
            'page'      : chunks[i]['page'],
            'char_count': chunks[i]['char_count'],
            'source'    : 'climate.pdf',
            'pdf_lib'   : 'pypdf',
        }
    }
    for i in range(len(chunks))
]
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors in {time.time()-t0:.1f}s")


## Step 7 — Conversation Memory

`ConversationMemory` stores `{role, content}` turns keyed by `session_id`.
The `resolve()` method detects pronouns and injects the last two turns as
inline context, producing a self-contained query for embedding.


In [ ]:
class ConversationMemory:
    """Session-keyed conversation store with pronoun-based query resolution."""

    PRONOUNS = {'it', 'its', 'they', 'them', 'their',
                'this', 'that', 'these', 'those'}

    def __init__(self, window: int = MEMORY_WINDOW):
        self._store: Dict[str, List[Dict]] = {}
        self.window = window

    def add(self, session_id: str, role: str, content: str) -> None:
        if session_id not in self._store:
            self._store[session_id] = []
        self._store[session_id].append({'role': role, 'content': content})

    def get_turns(self, session_id: str) -> List[Dict]:
        return self._store.get(session_id, [])[-self.window:]

    def has_pronouns(self, query: str) -> bool:
        words = set(re.findall(r'\b\w+\b', query.lower()))
        return bool(words & self.PRONOUNS)

    def resolve(self, session_id: str, query: str) -> str:
        """Prepend last two turns as context when pronouns are detected."""
        if not self.has_pronouns(query):
            return query
        turns = self.get_turns(session_id)
        if not turns:
            return query
        ctx_lines = []
        for t in turns[-2:]:
            snippet = t['content'][:250].replace('\n', ' ')
            ctx_lines.append(f"{t['role'].upper()}: {snippet}")
        ctx = '\n'.join(ctx_lines)
        return f"[Prior context:\n{ctx}\n]\nCurrent question: {query}"

    def format_history(self, session_id: str) -> str:
        turns = self.get_turns(session_id)
        if not turns:
            return "(No prior conversation)"
        return '\n'.join(
            f"{t['role'].upper()}: {t['content'][:300]}" for t in turns)

    def turn_count(self, session_id: str) -> int:
        return len(self._store.get(session_id, []))


print("ConversationMemory defined.")


## Step 8 — Memory-Augmented RAG Pipeline

Pipeline per turn:
1. `memory.resolve()` — detect pronouns, inject prior context into query
2. Embed resolved query → vector search → retrieve top-K chunks
3. Build prompt: `conversation history + retrieved context + question`
4. LLM generates answer
5. Store `(user, question)` and `(assistant, answer)` in memory


In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about a climate document.

Rules:
- Answer ONLY from the provided document context.
- Use the conversation history to interpret pronoun references (it, they, this, these, etc.).
- If a question references something not in the context, say so explicitly.
- Be concise and precise.
"""


def memory_augmented_rag(
    question: str,
    session_id: str,
    memory: ConversationMemory,
    verbose: bool = True,
) -> Dict:
    t0 = time.time()

    # 1. Resolve pronouns using memory
    resolved         = memory.resolve(session_id, question)
    pronoun_resolved = resolved != question

    # 2. Embed resolved query and retrieve
    q_vec   = embed_text(resolved)
    hits    = vs.vector_search(q_vec, top_k=DEFAULT_TOP_K)
    context = '\n\n'.join(
        f"[{i+1}] page={h['metadata'].get('page','?')}\n{h['text']}"
        for i, h in enumerate(hits))

    # 3. Build user message with history + retrieved context
    history  = memory.format_history(session_id)
    user_msg = f"""Conversation history:
{history}

Document context:
{context}

Question: {question}"""

    # 4. Generate answer
    answer  = call_llm(SYSTEM_PROMPT, user_msg)
    latency = (time.time() - t0) * 1000

    # 5. Store turn in memory
    memory.add(session_id, 'user',      question)
    memory.add(session_id, 'assistant', answer)

    if verbose:
        tag    = ' [pronoun resolved]' if pronoun_resolved else ''
        qa_num = memory.turn_count(session_id) // 2
        print(f"Q{tag}: {question}")
        print(f"A: {answer[:500]}")
        print(f"   [{latency:.0f}ms | {len(hits)} chunks | turn {qa_num}]")
        print()

    return {
        'question'        : question,
        'resolved_query'  : resolved,
        'pronoun_resolved': pronoun_resolved,
        'answer'          : answer,
        'n_chunks'        : len(hits),
        'latency_ms'      : latency,
    }


print("memory_augmented_rag() defined.")


## Step 9 — Multi-Turn Demo

Five questions where turns 2, 4, and 5 use pronouns referencing entities
from prior answers, exercising the resolution mechanism.


In [ ]:
SESSION = "demo_session_1"
memory  = ConversationMemory(window=MEMORY_WINDOW)

questions = [
    # Turn 1 — standalone
    "What is ensemble forecasting in weather prediction?",
    # Turn 2 — pronoun 'its' → ensemble forecasting
    "What are its main advantages over deterministic forecasting?",
    # Turn 3 — new topic
    "What observation data sources does the document mention?",
    # Turn 4 — pronoun 'they' → data sources
    "How are they collected and processed before being fed into models?",
    # Turn 5 — pronoun 'these' → data sources / both topics
    "How does ensemble forecasting use these data sources?",
]

results = []
print("=" * 70)
for q in questions:
    r = memory_augmented_rag(q, SESSION, memory, verbose=True)
    results.append(r)
    print("-" * 70)


## Step 9b — Resolution Analysis

In [ ]:
resolved_count = sum(1 for r in results if r['pronoun_resolved'])
print(f"Session              : {SESSION}")
print(f"Q&A turns            : {len(results)}")
print(f"Pronoun resolutions  : {resolved_count}/{len(results)}")
print(f"Avg latency          : {sum(r['latency_ms'] for r in results)/len(results):.0f}ms")
print(f"Avg chunks/Q         : {sum(r['n_chunks'] for r in results)/len(results):.1f}")
print()
print("Pronoun resolution detail:")
for r in results:
    tag = "✓ resolved" if r['pronoun_resolved'] else "  direct  "
    print(f"  [{tag}] {r['question'][:65]}")


## Step 10 — Summary

### What Memory-Augmented RAG does differently

| Dimension | Standard RAG | Memory-Augmented RAG |
|-----------|-------------|----------------------|
| Query scope | Single-turn | Multi-turn with history |
| Pronoun handling | Fails silently | Resolved from prior context |
| Retrieval query | Raw user question | Pronoun-resolved enriched query |
| Generation prompt | Chunks only | Chunks + conversation history |
| Session state | None | Per-session memory store |

### Pronoun resolution heuristic

```
if query contains {it, its, they, them, their, this, that, these, those}:
    prepend last 2 turns as [Prior context: ...]
    embed the enriched string instead of the raw query
```

### pypdf vs other PDF frameworks used in this series

| Library | API style | Layout info | Best for |
|---------|-----------|-------------|----------|
| pypdf | `page.extract_text()` | None | Clean text PDFs, simplicity |
| pdfplumber | `page.extract_words()` | BBox per word | Tables, density analysis |
| pymupdf `dict` | `get_text("dict")` | Span/font level | Font-based heading detection |
| pymupdf `blocks` | `get_text("blocks")` | Block-level + type flag | Fast block extraction |
| pdfminer.six | `LTTextBox` objects | Layout zones (y0) | Zone-based page analysis |

> **Next: [24 — Multi-Turn Conversational RAG](24_Multi_Turn_Conversational_RAG.ipynb)** — dual vector stores
> (document KB + conversation history); retrieve relevant past turns alongside document chunks.


In [ ]:
print(f"Collection '{COLLECTION_NAME}': {vs.count()} vectors indexed")
print(f"PDF framework : pypdf PdfReader — page.extract_text()")
print(f"RAG pattern   : Memory-Augmented RAG — pronoun resolution + history-grounded generation")
print(f"Memory        : ConversationMemory (window={MEMORY_WINDOW} turns, session-keyed)")
print()
print("Notebook 23 — Memory-Augmented RAG complete.")
